# Notebook 3: Promote and Decay

The gate decided what gets in. Promote and Decay decide what happens to a memory afterwards, and we run them on a simulated clock. There are two TODOs, each with a check cell, and the clock walk at the end is run-and-explain. Everything here runs on a scratch copy of the handbook. Your own store is read for a count and never written to.

In [1]:
# Make `import memory` work whether Jupyter opened the repo root or the notebooks folder.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "memory" / "__init__.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import contextlib
import io

with contextlib.redirect_stderr(io.StringIO()):  # a noisy third-party import warning we do not need in the room
    import weaviate.classes.init  # noqa: F401

from collections import Counter
from datetime import timedelta

from memory import Candidate, capture, collection_name, connect, describe_tier, fetch_all, score_candidate
from memory.lifecycle import (
    PROMOTION_RULES,
    SESSION_UNUSED_DAYS,
    TEAM_UNUSED_DAYS,
    corroborate,
    decay,
    decay_decision,
    endorse,
    promote,
    promote_all,
    promotion_decision,
    renew,
)
from memory.reference import reference_gate
from memory.schema import LIFESPAN_DAYS, REVIEW_DAYS, now_utc
from weaviate.classes.query import Filter

print(describe_tier())

WEAVIATE_MODE=docker (tier 1, local Docker container), collection MemoryRecord


In [2]:
client = connect()
if not client.collections.exists(collection_name()):
    raise RuntimeError("no collection yet. Run notebook 0 first; it creates and seeds the store.")
mine = client.collections.get(collection_name())  # your store: read for a count, never written to here
start_count = len(fetch_all(mine, status="active"))
if start_count == 0:
    raise RuntimeError("no active records. Run the reset cell in notebook 0 to seed the handbook.")
print(f"your store holds {start_count} active records; this notebook will not change that")

from memory import ensure_collection
from memory.seed import seed_store

# A scratch copy of the handbook for the clock walk. Same 50 facts, same dates, separate collection,
# so decay() can run for real without touching anything you admitted in notebooks 1 and 2.
SCRATCH = collection_name("MemoryLifecycle")
OWNER_TAG = f"scratch copy owned by {collection_name()}"  # carries ATTENDEE_HANDLE on the cloud tier


def our_scratch(name):
    """The scratch collection, created and tagged if missing. Never drops a collection: if one exists under
    this name without our tag it is somebody else's and we stop. The tag guards against accidents, not
    against a hostile client: anyone holding the shared cloud key can delete any collection outright, so
    no in-band check could stop them, and the README says the cloud tier is a classroom convenience, not
    isolation. On that tier the handle is the identity, so two people on one handle share one scratch copy."""
    if client.collections.exists(name):
        coll = client.collections.get(name)
        desc = coll.config.get().description or ""
        if desc != OWNER_TAG:
            raise RuntimeError(f"{name} exists and is not this notebook's scratch copy ({desc!r}); refusing to touch it")
        return coll
    coll = ensure_collection(client, name=name)
    coll.config.update(description=OWNER_TAG)
    return coll


def wipe(coll) -> int:
    """Delete every object in a collection we own, whatever its shape, by id. The collection itself stays."""
    ids = [str(o.uuid) for o in coll.iterator()]
    removed = 0
    for i in range(0, len(ids), 500):
        removed += int(coll.data.delete_many(where=Filter.by_id().contains_any(ids[i:i + 500])).successful)
    return removed


def claim_scratch(name):
    """A fresh scratch copy: ours, emptied, reseeded with the handbook. The one place that seeds it."""
    coll = our_scratch(name)
    wipe(coll)
    seed_store(coll)
    return coll


def release_scratch(name):
    """Empty the scratch copy and leave the tagged, empty collection for the next run to reclaim."""
    if not client.collections.exists(name):
        print(f"{name} does not exist; nothing to clean")
        return
    print(f"emptied {name}: {wipe(our_scratch(name))} records removed; the tagged collection stays for reuse")


collection = claim_scratch(SCRATCH)
print(f"seeded {len(fetch_all(collection))} handbook records into the scratch collection {SCRATCH}")

your store holds 50 active records; this notebook will not change that
seeded 50 handbook records into the scratch collection MemoryLifecycle


## The rules, on one screen

Three small tables in `memory/lifecycle.py` and `memory/schema.py` are the whole policy. Every result in this notebook traces back to one of these numbers.

In [3]:
print("PROMOTION_RULES   what it takes to move up into a tier")
for tier, rule in PROMOTION_RULES.items():
    print(
        f"  -> {tier:7s} {rule['min_corroboration']} distinct sources, confidence >= {rule['min_confidence']}, "
        f"{'human endorsement required' if rule['needs_endorsement'] else 'no endorsement needed'}"
    )
print("\nLIFESPAN_DAYS     how long a record lives in a tier before it expires (None: never on its own)")
print("REVIEW_DAYS       when a human is asked to look at it again")
for tier in LIFESPAN_DAYS:
    print(f"  {tier:8s} lifespan {str(LIFESPAN_DAYS[tier]):5s} review every {REVIEW_DAYS[tier]} days")
print(f"\nusage: a session memory unread for {SESSION_UNUSED_DAYS} days expires; a team memory idle for {TEAM_UNUSED_DAYS} days is flagged")

PROMOTION_RULES   what it takes to move up into a tier
  -> team    2 distinct sources, confidence >= 0.6, no endorsement needed
  -> org     3 distinct sources, confidence >= 0.7, human endorsement required

LIFESPAN_DAYS     how long a record lives in a tier before it expires (None: never on its own)
REVIEW_DAYS       when a human is asked to look at it again
  session  lifespan 7     review every 3 days
  team     lifespan 90    review every 30 days
  org      lifespan None  review every 180 days

usage: a session memory unread for 3 days expires; a team memory idle for 60 days is flagged


## Three memories go in

They go through the reference gate from notebook 2, at time zero. All three start in the session tier. Then the agent reads one of them back with usage tracking on, so Decay will know it was used.

In [4]:
from memory.evaluate import load_labeled
from memory.retrieve import recall, show

T0 = now_utc()


def at(days: int):
    """The simulated clock: T0 plus some days."""
    return T0 + timedelta(days=days)


labeled = {item["id"]: item for item in load_labeled()}
northgate = capture(
    text="Northgate Logistics asked for a 24 hour heads-up before freight deliveries.",
    source="agent:harbor", source_type="agent", context="support-session-9001",
    confidence=0.85, topic="customer", scope="customer:northgate-logistics",
)
halvorsen = Candidate.from_dict(labeled["good-01"])  # Halvorsen invoices, heard by agent:harbor
riverbend = Candidate.from_dict(labeled["good-02"])  # Riverbend fiscal year, from user:carla.mendes
demo = {"northgate": northgate, "halvorsen": halvorsen, "riverbend": riverbend}
gate = reference_gate()


def admit_three(collection):
    """Admit the three demo candidates at T0 and read Riverbend back once. Returns name -> uuid."""
    uuids = {}
    for name, cand in demo.items():
        cand.candidate_id = name
        d = gate.admit(collection, cand, score_candidate(collection, cand), now=T0)
        print(f"{name:10s}", d)
        assert d.outcome == "written", f"{name} was not written: {d.reason}"
        uuids[name] = d.record_uuid
    # The query is a shipped text, so the cached vector is used and the embedding model is not needed here.
    recall(collection, riverbend.text, scope="customer:riverbend-schools",
           tiers=["session"], limit=1, track_usage=True, now=T0)
    return uuids


uuids = admit_three(collection)


def record(name):
    obj = collection.query.fetch_object_by_id(uuids[name])
    return {"uuid": str(obj.uuid), **obj.properties}


def board(title):
    print(f"\n{title}")
    for name in demo:
        r = record(name)
        print(f"  {name:10s} {r['tier']:8s} {r['status']:13s} x{r['corroboration_count']}  used {r['use_count']}  "
              f"review {r['review_at'].date()}  expires {r['expires_at'].date() if r['expires_at'].year < 2999 else 'never'}")


board("at T0:")
assert record("riverbend")["use_count"] == 1, "the tracked recall should have marked the Riverbend record as used"

northgate  ALLOW [written] 'Northgate Logistics asked for a 24 hour heads-up before freight delive'
halvorsen  ALLOW [written] 'Halvorsen Dental wants all invoices sent to accounts@halvorsendental.e'
riverbend  ALLOW [written] "Riverbend School District's fiscal year starts July 1, so their purcha"

at T0:
  northgate  session  active        x1  used 0  review 2026-10-27  expires 2026-10-31
  halvorsen  session  active        x1  used 0  review 2026-10-27  expires 2026-10-31
  riverbend  session  active        x1  used 1  review 2026-10-27  expires 2026-10-31


## Promote, and TODO 1

`promote()` applies the rules to one record and tells you what it did and why. `corroborate()` adds a distinct source. `endorse()` puts a named human on the record. Your job is to take Northgate all the way to org, which needs three distinct sources and an endorser. Weaviate docs: [updating objects](https://docs.weaviate.io/weaviate/manage-objects/update).

In [5]:
uid = uuids["northgate"]
print("1 source:    ", promote(collection, uid, now=T0))

corroborate(collection, uid, "user:ben.oyelaran", now=T0)
print("2 sources:   ", promote(collection, uid, now=at(1)))

r = corroborate(collection, uid, "agent:harbor", now=at(1))  # harbor already vouched; this does not count
print("harbor again: corroboration_count still", r["corroboration_count"], "from", r["corroborated_by"])

# TODO 1a: corroborate from "user:carla.mendes" at day 2
print("day 2:       ", promote(collection, uid, now=at(2)))
# TODO 1b: endorse by "user:aisha.karim" at day 3
print("day 3:       ", promote(collection, uid, now=at(3)))

r = record("northgate")
print(f"\nnorthgate is in {r['tier']} (org when TODO 1 is done), endorsed by {r['endorsed_by'] or 'nobody'}, "
      f"review {r['review_at'].date()}, expires {'never' if r['expires_at'].year == 2999 else r['expires_at'].date()}")

1 source:     (None, 'needs 2 distinct sources, has 1')
2 sources:    ('team', '2 sources, confidence 0.85')
harbor again: corroboration_count still 2 from ['agent:harbor', 'user:ben.oyelaran']
day 2:        (None, 'needs 3 distinct sources, has 2')
day 3:        (None, 'needs 3 distinct sources, has 2')

northgate is in team (org when TODO 1 is done), endorsed by nobody, review 2026-11-24, expires 2027-01-23


`promotion_decision()` is the pure half of `promote()`: a record goes in, the next tier and a reason come out, and no database is touched. That is the function to unit test.

In [6]:
for name in demo:
    print(f"{name:10s}", promotion_decision(record(name)))

print("\npromote_all at day 3, one pass over every active record:")
moved = [row for row in promote_all(collection, now=at(3)) if row["to"]]
print(f"  {len(moved)} promoted; the handbook is already at org and the other two have one source each")

northgate  (None, 'needs 3 distinct sources, has 2')
halvorsen  (None, 'needs 2 distinct sources, has 1')
riverbend  (None, 'needs 2 distinct sources, has 1')

promote_all at day 3, one pass over every active record:


  0 promoted; the handbook is already at org and the other two have one source each


## Decay, and TODO 2

`decay_decision()` picks a new status for one record, or leaves it alone. There are four rules, in order: the lifespan ended, a session memory was never used, a team memory went idle, or a review came due. It never deletes. The version below is missing the second rule: a session record with a `use_count` of 0 that has been in the tier for at least `SESSION_UNUSED_DAYS` expires.

In [7]:
def my_decay_decision(record, now):
    tier = record["tier"]
    expires_at = record.get("expires_at")
    review_at = record.get("review_at")
    last_used = record.get("last_used") or record.get("captured_at")
    use_count = int(record.get("use_count") or 0)
    age = now - (record.get("promoted_at") or record["captured_at"])  # time in the current tier

    if tier != "org" and expires_at and now >= expires_at:
        return "expired", f"{tier} lifespan ended {expires_at.date()}"
    # TODO 2: if tier is "session", use_count is 0 and age >= timedelta(days=SESSION_UNUSED_DAYS),
    #         return ("expired", f"never used in {age.days} days")
    if tier == "team" and last_used and now - last_used >= timedelta(days=TEAM_UNUSED_DAYS):
        return "needs_review", f"not used since {last_used.date()}"
    if review_at and now >= review_at:
        return "needs_review", f"review was due {review_at.date()}"
    return None, "healthy"


for name in demo:
    print(f"{name:10s} at day 4:", my_decay_decision(record(name), at(4)))
print("\n(halvorsen reads 'expired, never used in 4 days' when TODO 2 is done)")

northgate  at day 4: (None, 'healthy')
halvorsen  at day 4: ('needs_review', 'review was due 2026-10-27')
riverbend  at day 4: ('needs_review', 'review was due 2026-10-27')

(halvorsen reads 'expired, never used in 4 days' when TODO 2 is done)


### Check your work

This compares yours against `decay_decision` in `memory/lifecycle.py`, on every active record in the scratch store, at day 4 and at day 200.

In [8]:
def compare():
    rows = fetch_all(collection, status="active")
    mismatches = 0
    for days in (4, 200):
        for r in rows:
            mine, theirs = my_decay_decision(r, at(days)), decay_decision(r, at(days))
            if mine != theirs:
                mismatches += 1
                print(f"MISMATCH day {days:3d}  yours={mine[0]!s:12s} ref={theirs[0]!s:12s} {r['text'][:50]}")
    print(f"{len(rows)} records x 2 days, {mismatches} mismatches")
    return mismatches == 0


compare()

MISMATCH day   4  yours=needs_review ref=expired      Halvorsen Dental wants all invoices sent to accoun
53 records x 2 days, 1 mismatches


False

## The clock walk

Now the real `decay()`: one pass over every active record at day 4, 10, 100 and 200. Statuses change, nothing is deleted, and `recall()` only reads `active`.

In [9]:
def tick(days):
    actions = decay(collection, now=at(days))
    by_uuid = {a["uuid"]: a for a in actions}
    print(f"decay at day {days}: {dict(Counter(a['action'] for a in actions))}")
    for name in demo:
        a = by_uuid.get(uuids[name])
        print(f"  {name:10s} {a['action']:13s} {a['reason']}" if a else f"  {name:10s} (not active, skipped)")
    return actions


actions = tick(4)

decay at day 4: {'keep': 51, 'expired': 1, 'needs_review': 1}
  northgate  keep          healthy
  halvorsen  expired       never used in 4 days
  riverbend  needs_review  review was due 2026-10-27


Day 4. Halvorsen expired because nobody read it. Riverbend was read, so it got flagged for review instead. Northgate and the handbook hold. A flag is a question for a human, and `renew()` only accepts a `needs_review` record. An expired one has to come back in through the gate.

In [10]:
uid = uuids["riverbend"]
r = renew(collection, uid, by="user:carla.mendes", now=at(4))
print("renewed:  ", r["status"], "review", r["review_at"].date(), "expires", r["expires_at"].date())

try:
    renew(collection, uuids["halvorsen"], by="user:carla.mendes", now=at(4))
except ValueError as e:
    print("halvorsen:", e)

corroborate(collection, uid, "agent:harbor", now=at(4))
corroborate(collection, uid, "user:ben.oyelaran", now=at(4))
print("promote:  ", promote(collection, uid, now=at(4)))
board("at day 4, after the human acted:")

renewed:   active review 2026-10-31 expires 2026-11-04
halvorsen: only needs_review records can be renewed; this one is expired
promote:   ('team', '3 sources, confidence 0.90')

at day 4, after the human acted:
  northgate  team     active        x2  used 0  review 2026-11-24  expires 2027-01-23
  halvorsen  session  expired       x1  used 0  review 2026-10-27  expires 2026-10-31
  riverbend  team     active        x3  used 1  review 2026-11-27  expires 2027-01-26


In [11]:
actions = tick(10)
actions = tick(100)

decay at day 10: {'keep': 52}
  northgate  keep          healthy
  halvorsen  (not active, skipped)
  riverbend  keep          healthy
decay at day 100: {'keep': 50, 'expired': 2}
  northgate  expired       team lifespan ended 2027-01-23
  halvorsen  (not active, skipped)
  riverbend  expired       team lifespan ended 2027-01-26


Day 10, everything holds. Day 100, the Riverbend team record's 90-day lifespan ran out. One more jump.

In [12]:
actions = tick(200)

decay at day 200: {'needs_review': 50}
  northgate  (not active, skipped)
  halvorsen  (not active, skipped)
  riverbend  (not active, skipped)


Day 200, with Northgate in org: 51 records flagged and nothing expired. Northgate came due at day 183 and the handbook at day 150. Org memories never vanish on a timer, and they never stay true unexamined either. Without TODO 1, Northgate would have died in the team tier at day 91 instead.

In [13]:
r = record("northgate")
if r["status"] == "needs_review":
    r = renew(collection, uuids["northgate"], by="user:aisha.karim", now=at(200))
    print(f"northgate renewed by {r['endorsed_by']}: {r['status']}, next review {r['review_at'].date()}")
else:
    print(f"northgate is {r['status']} in the {r['tier']} tier. Without TODO 1 it never reached org, so its 90-day "
          "team lifespan ran out at day 91. Do TODO 1 and rerun from the clock walk, or run the fell-behind cell.")
print("\nstore by status at day 200:", dict(Counter(x["status"] for x in fetch_all(collection))))

northgate is expired in the team tier. Without TODO 1 it never reached org, so its 90-day team lifespan ran out at day 91. Do TODO 1 and rerun from the clock walk, or run the fell-behind cell.

store by status at day 200: {'needs_review': 50, 'expired': 3}


## Fell behind, or something broke?

This cell swaps in the reference `decay_decision`, rebuilds and reseeds the scratch copy, re-admits the three memories and walks Northgate to org with the reference calls. Then carry on from the clock walk, or go to the cleanup below.

In [14]:
from memory.lifecycle import decay_decision as my_decay_decision  # noqa: F811

collection = claim_scratch(SCRATCH)
uuids = admit_three(collection)
uid = uuids["northgate"]
corroborate(collection, uid, "user:ben.oyelaran", now=T0)
promote(collection, uid, now=at(1))
corroborate(collection, uid, "user:carla.mendes", now=at(2))
endorse(collection, uid, "user:aisha.karim", now=at(3))
print("northgate:", promote(collection, uid, now=at(3)))
assert compare(), "the reference decay_decision should match itself"
board("rebuilt, at day 3:")

northgate  ALLOW [written] 'Northgate Logistics asked for a 24 hour heads-up before freight delive'
halvorsen  ALLOW [written] 'Halvorsen Dental wants all invoices sent to accounts@halvorsendental.e'
riverbend  ALLOW [written] "Riverbend School District's fiscal year starts July 1, so their purcha"


northgate: ('org', '3 sources, confidence 0.85, endorsed by user:aisha.karim')
53 records x 2 days, 0 mismatches

rebuilt, at day 3:
  northgate  org      active        x3  used 0  review 2027-04-25  expires never
  halvorsen  session  active        x1  used 0  review 2026-10-27  expires 2026-10-31
  riverbend  session  active        x1  used 1  review 2026-10-27  expires 2026-10-31


## Leave the store as you found it

The scratch copy gets emptied. The tagged collection stays, so the next run can reclaim it without dropping anything. Your store was never written to.

In [15]:
release_scratch(SCRATCH)

emptied MemoryLifecycle: 53 records removed; the tagged collection stays for reuse


## What was Weaviate-specific here, and what was not

Weaviate-specific: `collection.data.update` for every status, tier and date change, `fetch_object_by_id`, and the scratch collection, which is created once and emptied with a filtered delete. Portable: the three rule tables, `promotion_decision()` and `decay_decision()` as pure functions, the rule that the top tier needs a named human, the rule that the lifecycle never deletes, and the simulated clock.

## Green check

In [16]:
count = len(fetch_all(mine, status="active"))
assert count == start_count, (
    f"your store started with {start_count} active records and now has {count}. This notebook never writes to "
    "it, so something else changed it; run the reset cell in notebook 0 before notebook 4."
)
assert not client.collections.exists(SCRATCH) or len(fetch_all(collection)) == 0, f"{SCRATCH} still holds records; run the cleanup cell above"
print(f"\u2705 {describe_tier()}")
print(f"\u2705 {count} records in your store, untouched. Scratch copy emptied. Notebook 4 measures how well all of this works.")
client.close()

✅ WEAVIATE_MODE=docker (tier 1, local Docker container), collection MemoryRecord
✅ 50 records in your store, untouched. Scratch copy emptied. Notebook 4 measures how well all of this works.
